In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nlp-getting-started/sample_submission.csv
/kaggle/input/nlp-getting-started/train.csv
/kaggle/input/nlp-getting-started/test.csv


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.metrics import F1Score
from keras import regularizers

**Load Dataset**

In [3]:
train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test= pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

**Data analysis**

In [4]:
print(train.shape, test.shape)
print(train.columns, test.columns)

(7613, 5) (3263, 4)
Index(['id', 'keyword', 'location', 'text', 'target'], dtype='object') Index(['id', 'keyword', 'location', 'text'], dtype='object')


In [5]:
print(train.keyword.value_counts())
print(f"Number of NA: {train.keyword.isna().sum()}")

keyword
fatalities               45
armageddon               42
deluge                   42
harm                     41
damage                   41
                         ..
forest%20fire            19
epicentre                12
threat                   11
inundation               10
radiation%20emergency     9
Name: count, Length: 221, dtype: int64
Number of NA: 61


In [6]:
tokenizer = Tokenizer(num_words = 3000, oov_token="<OOV>")
tokenizer.fit_on_texts(train["text"])
word_index = tokenizer.word_index

train_sequences = tokenizer.texts_to_sequences(train["text"])
train_padded = pad_sequences(train_sequences, padding = "post")

test_sequences = tokenizer.texts_to_sequences(test["text"])
test_padded = pad_sequences(test_sequences, padding = "post", maxlen= train_padded.shape[1])

In [7]:
print(train_padded.shape, test_padded.shape)

(7613, 33) (3263, 33)


In [8]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(3000, 33),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(33, return_sequences = True, kernel_regularizer=regularizers.l2(200))),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(33, return_sequences = True, kernel_regularizer=regularizers.l2(5))),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(33, kernel_regularizer=regularizers.l2(5))),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=[F1Score(threshold = 0.5)])#'accuracy'

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_f1_score', patience=10, restore_best_weights=True)

model.fit(train_padded, train["target"], 
          validation_split =0.2, 
          epochs=30, 
          batch_size=128, 
          callbacks=[early_stopping])


Epoch 1/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - f1_score: 0.0434 - loss: 19119.8535 - val_f1_score: 0.0000e+00 - val_loss: 10045.4258
Epoch 2/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - f1_score: 0.1799 - loss: 8204.3340 - val_f1_score: 0.6688 - val_loss: 4048.5227
Epoch 3/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - f1_score: 0.7252 - loss: 3253.1196 - val_f1_score: 0.6606 - val_loss: 1501.6448
Epoch 4/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - f1_score: 0.7632 - loss: 1184.2279 - val_f1_score: 0.6827 - val_loss: 506.6357
Epoch 5/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - f1_score: 0.7821 - loss: 391.6126 - val_f1_score: 0.5977 - val_loss: 154.6004
Epoch 6/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - f1_score: 0.7653 - loss: 117.0448 - val_f1_score: 0.6764 - val_loss: 42.8084
Epoch 7/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - f1_score: 0.6984 - loss: 31.7857 - val_f1_score: 0.0000e+00 - val_loss: 11.0435
Epoch 8/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - f1_score: 0.0849

In [11]:
y_pred = model.predict(test_padded)
y_submit = []
for val in y_pred:
    if val >= 0.5:
        y_submit += [1]
    else:
        y_submit += [0]

102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


In [12]:
test_id = pd.DataFrame(test["id"])
merged_df = pd.concat([test_id, pd.DataFrame(y_submit, columns=["target"])], axis=1)

In [13]:
merged_df.to_csv("output.csv", index = False)